# 05 - Model Training (XGBoost / LightGBM)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, pickle, json
from src.utils import FEATURES_DATA_PATH, TARGET_COL, set_seed, RANDOM_SEED
from src.forecasting import (chronological_split, _encode_categoricals,
                              get_model_feature_cols, fit_xgboost, fit_sklearn_gbm)
from src.evaluation import evaluate

set_seed()
df = pd.read_csv(FEATURES_DATA_PATH, parse_dates=['date'])
train, val, test, meta = chronological_split(df)
train_enc, val_enc, test_enc = _encode_categoricals(train, val, test)
feat_cols = get_model_feature_cols()

X_train, y_train = train_enc[feat_cols], train_enc[TARGET_COL]
X_val, y_val = val_enc[feat_cols], val_enc[TARGET_COL]
X_test, y_test = test_enc[feat_cols], test_enc[TARGET_COL]
print(f"{len(feat_cols)} features: {feat_cols}")

27 features: ['lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28', 'day_of_week', 'week_of_year', 'month', 'quarter', 'year', 'is_weekend', 'promotion', 'price', 'inventory_level', 'discount_pct', 'price_vs_competitor', 'price_vs_competitor_pct', 'cat_expanding_mean', 'store_id_enc', 'product_id_enc', 'category_enc', 'region_enc', 'weather_enc']


In [2]:
model = fit_xgboost(X_train, y_train, X_val, y_val)
print("Trained model: XGBoost")

Trained model: XGBoost


In [3]:
pred_test = model.predict(X_test)
ml_metrics = evaluate(y_test, pred_test)
ml_metrics

{'MAE': 69.018, 'RMSE': 88.161, 'MAPE_%': 246.468, 'WAPE_%': 50.56}

In [4]:
os.makedirs('../models', exist_ok=True)
with open('../models/demand_forecast_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('../models/feature_cols.json', 'w') as f:
    json.dump(feat_cols, f)

test_out = test_enc[['date','store_id','product_id','category','region', TARGET_COL,'inventory_level']].copy()
test_out['pred_demand'] = np.clip(pred_test, 0, None)
test_out.to_csv('../data/processed/test_predictions.csv', index=False)
print("Saved model, feature list, and test predictions.")

Saved model, feature list, and test predictions.


In [5]:
import json

with open("../data/processed/xgboost_metrics.json", "w") as f:
    json.dump({"XGBoost (gradient boosted trees)": ml_metrics}, f, indent=2)

print("Saved XGBoost metrics:", ml_metrics)

Saved XGBoost metrics: {'MAE': 69.018, 'RMSE': 88.161, 'MAPE_%': 246.468, 'WAPE_%': 50.56}
